<a href="https://colab.research.google.com/github/klavarihards/Markets-Strategies/blob/main/portfolio-engine/notebooks/05_product_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Portfolio Engine, Phase 5: The Client Product

What a retail client and their advisor actually see. The client never sees a covariance matrix: they answer six questions and get a one-page report.

The product runs the frozen portfolio of each profile (`configs/final_choice.yaml`) exactly as backtested, on all the data up to today. It is only allowed to load the full data because the one-time hold-out evaluation has already been run and logged.

*Educational research project; the clients below are hypothetical.*

In [1]:
# Setup: clone the repo (in Colab) and install the package, plus the libraries used for cross-checks.
import os, sys

IN_COLAB = "google.colab" in sys.modules
BRANCH = "claude/eloquent-fermat-i94z76"     # change to "main" once the branch is merged

if IN_COLAB:
    os.chdir("/content")
    if not os.path.exists("Markets-Strategies"):
        !git clone -q -b {BRANCH} https://github.com/klavarihards/Markets-Strategies.git
    else:
        !git -C Markets-Strategies fetch -q origin {BRANCH} && git -C Markets-Strategies checkout -q {BRANCH} && git -C Markets-Strategies pull -q origin {BRANCH}
    os.chdir("Markets-Strategies/portfolio-engine")
    !pip install -q -e ".[download,product]"
    sys.path.insert(0, "src")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

# In Colab, keep data snapshots and results on Google Drive: a Colab runtime's own files
# disappear when it restarts, and the snapshot must stay the same for every later run.
# The repo's data/snapshots and results folders become links to Drive, so nothing else changes.
USE_DRIVE = IN_COLAB
if USE_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    store = "/content/drive/MyDrive/portfolio-engine"
    for local, remote in [("data/snapshots", f"{store}/snapshots"), ("results", f"{store}/results")]:
        os.makedirs(remote, exist_ok=True)
        if not os.path.islink(local):
            if os.path.isdir(local):
                shutil.copytree(local, remote, dirs_exist_ok=True)   # keep anything already there
                shutil.rmtree(local)
            os.makedirs(os.path.dirname(local) or ".", exist_ok=True)
            os.symlink(remote, local)
    print("Snapshots and results are stored in", store)

Working directory: /home/user/Markets-Strategies/portfolio-engine


In [2]:
import json
from datetime import date
import pandas as pd
import matplotlib.pyplot as plt

from portfolio_engine.config import load_config
from portfolio_engine.data import read_snapshot, resolve_snapshot
from portfolio_engine.product import Advisor, Answers
from portfolio_engine.product.report import render_pdf
from portfolio_engine.product.implementation import tracking_report

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
advisor = Advisor("configs/product.yaml")       # builds the three frozen portfolios once
print("Data until", advisor.data.returns.index[-1].date(), "| snapshot", advisor.data.snapshot_id)
print(json.dumps(advisor.choice["portfolios"], indent=1))

Data until 2026-10-22 | snapshot 570bdef9e8ea
{
 "conservative": {
  "method": "min_cvar",
  "overlay": "all"
 },
 "balanced": {
  "method": "min_cvar",
  "overlay": "regime_hmm"
 },
 "growth": {
  "method": "black_litterman",
  "overlay": "all"
 }
}


## Three hypothetical clients

Each answers the same six questions. The profile is the more cautious of risk *capacity* (horizon, income) and risk *tolerance* (what they would do after a 20% fall).

In [3]:
clients = {
    "Ms. Lin (home deposit top-up)": Answers(1_700_000, 2031, 1_000_000, 10_000, "stable", "sell"),
    "Mr. Chen (home deposit)": Answers(3_000_000, 2036, 600_000, 15_000, "stable", "hold"),
    "Ms. Wang (retirement)": Answers(12_000_000, 2056, 300_000, 10_000, "very_stable", "buy_more"),
}
rows, advice = [], {}
for name, a in clients.items():
    adv = advice[name] = advisor.advise(a)
    g, r = adv["goal"], adv["risk"]
    gfc = r["crises"]["global_financial_crisis"]
    rows.append({"client": name, "profile": adv["profile"]["profile"],
                 "portfolio": f"{adv['portfolio']['method']} + {adv['portfolio']['overlay']}",
                 "goal": f"NT${g['amount']:,.0f} by {g['year']}", "chance": f"{g['probability']:.0%}",
                 "median outcome": f"NT${g['median']:,.0f}", "bad case (1 in 20)": f"NT${g['p5']:,.0f}",
                 "+NT$2,000/month": f"{g['with_more_monthly']['probability']:.0%}",
                 "2 more years": f"{g['two_years_later']['probability']:.0%}",
                 "2008 replay": f"{gfc['max_drawdown']:.0%}", "review at": f"{r['review_trigger']:.1%}"})
pd.DataFrame(rows).set_index("client")

,profile,portfolio,goal,chance,median outcome,bad case (1 in 20),"+NT$2,000/month",2 more years,2008 replay,review at
client,,,,,,,,,,
Ms. Lin (home deposit top-up),conservative,min_cvar + all,"NT$1,700,000 by 2031",75%,"NT$1,779,895","NT$1,591,729",95%,100%,-11%,-7.5%
Mr. Chen (home deposit),balanced,min_cvar + regime_hmm,"NT$3,000,000 by 2036",68%,"NT$3,168,758","NT$2,607,040",89%,98%,-19%,-15.0%
Ms. Wang (retirement),growth,black_litterman + all,"NT$12,000,000 by 2056",76%,"NT$15,270,918","NT$8,723,651",88%,87%,-34%,-22.5%


In [4]:
pd.DataFrame({name: {h["name"]: h["weight"] for h in adv["portfolio"]["holdings"]} for name, adv in advice.items()}).fillna(0).style.format("{:.0%}")

,Ms. Lin (home deposit top-up),Mr. Chen (home deposit),Ms. Wang (retirement)
Taiwan stocks,7%,0%,5%
US stocks,17%,31%,37%
Other markets' stocks,6%,10%,28%
Medium-term bonds,35%,10%,11%
Cash,35%,40%,0%
Gold,0%,9%,10%
Long US Treasuries,0%,0%,9%


## The one-page report

Rendered from the same advice object (Jinja2 template → PDF); every number on it comes from the engine. Saved to `reports/`.

In [5]:
import os
os.makedirs("reports", exist_ok=True)
for name, adv in advice.items():
    fname = "reports/sample_report_" + name.split()[1].strip("().").lower() + ".pdf"
    open(fname, "wb").write(render_pdf(adv, name + " (hypothetical)"))
    print("Saved", fname)

<style> block 1: Ignoring CSS properties xhtml2pdf does not implement: border-collapse


Saved reports/sample_report_lin.pdf


<style> block 1: Ignoring CSS properties xhtml2pdf does not implement: border-collapse


Saved reports/sample_report_chen.pdf


<style> block 1: Ignoring CSS properties xhtml2pdf does not implement: border-collapse


Saved reports/sample_report_wang.pdf


## The API

The same engine behind a FastAPI service (`uvicorn portfolio_engine.product.api:app`). Here it is called in-process.

In [6]:
from fastapi.testclient import TestClient
from portfolio_engine.product.api import create_app
api = TestClient(create_app(advisor))
print(api.get("/health").json())
resp = api.post("/advice", json={"goal_amount": 3_000_000, "goal_year": 2036, "current_savings": 600_000,
                                 "monthly_contribution": 15_000, "income_stability": "stable",
                                 "reaction_to_20pct_fall": "hold", "avoid": ["gold"]})
body = resp.json()
print(resp.status_code, body["profile"]["profile"], f"{body['goal']['probability']:.0%}",
      [(h["name"], round(h["weight"], 3)) for h in body["portfolio"]["holdings"]])

{'status': 'ok', 'data_until': '2026-10-22', 'snapshot': '570bdef9e8ea'}


200 balanced 72% [('Taiwan stocks', 0.029), ('US stocks', 0.341), ("Other markets' stocks", 0.13), ('Medium-term bonds', 0.1), ('Cash', 0.4)]


## Implementation check: does 0050 behave like the research proxy?

The research used EWT (iShares MSCI Taiwan, in USD, converted to TWD); a Taiwan client would hold 0050 (Yuanta Taiwan Top 50). Monthly returns over the years both exist.

In [7]:
frames, _ = read_snapshot(resolve_snapshot(advisor.cfg["data"]["snapshot_dir"]))
tr = tracking_report(frames)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in tr.items() if k != "monthly"})
(1 + tr["monthly"]).cumprod().plot(figsize=(12, 4), logy=True, title="EWT in TWD vs 0050 (growth of 1 TWD, monthly)")
plt.grid(True, alpha=0.3); plt.show()

{'first': '2009-02-28', 'last': '2026-10-31', 'months': 213, 'correlation': 0.9183, 'tracking_error': 0.077, 'proxy_ann_return': 0.1626, 'fund_ann_return': 0.1924, 'proxy_max_drawdown': -0.2727, 'fund_max_drawdown': -0.2919, 'data_repairs': [{'date': '2014-01-02', 'ratio': 0.24936121009601955, 'split': '4-for-1'}], 'flagged_moves': {'2009-01-05': 0.1132, '2009-01-07': -0.1192, '2009-01-15': -0.1084, '2009-01-19': 0.1243, '2009-02-11': -0.1158, '2009-02-12': 0.119, '2009-02-19': 0.1655}}


## Findings

*Written after the run.*

- [RESULT]